# Face-Car: treino do classificador de sonolência (CNN)

Este notebook treina a rede que olha **um quadro da câmera** e devolve 3 probabilidades, **nesta ordem**:

| saída | classe | significado |
|---|---|---|
| 0 | `alerta` | motorista acordado |
| 1 | `sonolento` | olhos fechando, cabeça caindo |
| 2 | `bocejo` | bocejando |

**Contrato com o projeto** (o que `entrada/classificador.py` espera):

- entrada: imagem **RGB**, redimensionada para `TAMANHO × TAMANHO` **sem recortar**, pixels **de 0 a 255** (a normalização é uma camada do modelo);
- saída: `softmax` com 3 valores que somam 1, na ordem acima, que é a mesma de `classificador.classes` em `config/analise.json`.

Junto do `.keras`, o notebook salva `cnn_sonolencia.classes.json` com a ordem das classes. O projeto lê esse arquivo e **se recusa a rodar** se a configuração estiver em outra ordem.

**Como rodar no Colab:** *Ambiente de execução → Alterar o tipo → GPU (T4)*, ajuste a célula de **Configuração** e use *Executar tudo*. As células estão na ordem em que precisam rodar.

**Técnica usada (transfer learning):** a MobileNetV2 já foi treinada pelo Google com 1,4 milhão de fotos (ImageNet) e sabe extrair bordas, texturas, olhos e rostos. Ela vira a "base" congelada, e só uma camada nova no fim aprende a separar as 3 classes. Depois, opcionalmente, as últimas camadas da base são liberadas para um **ajuste fino**.

## 1. Configuração

Tudo o que muda de um treino para outro fica aqui.

- `CLASSES` liga **cada saída do modelo** à **subpasta do dataset** que tem as imagens dela. A **ordem das chaves** é a ordem das saídas. Se as pastas tiverem outro nome (ex.: `Drowsy`), mude só o valor, nunca a ordem.
- `PROPORCOES` divide as imagens em **treino** (a rede aprende), **validação** (decide quando parar) e **teste** (nota final, usado uma vez só no fim).
- `REGEX_PESSOA`: veja a seção 4. Se os nomes dos arquivos dizem de quem é a foto, preencha. Isso evita uma nota falsamente alta.

In [ ]:
# Saída do modelo → subpasta do dataset. A ORDEM é a das saídas e tem que ser a de
# classificador.classes em config/analise.json: alerta, sonolento, bocejo.
CLASSES = {
    "alerta": "alerta",
    "sonolento": "sonolento",
    "bocejo": "bocejo",
}

PASTA_DRIVE = "/content/drive/MyDrive/face_car/dataset"  # dataset original, no Google Drive
PASTA_LOCAL = "/content/dataset"                         # cópia no disco do Colab (muito mais rápida)
PASTA_SAIDA = "/content/drive/MyDrive/face_car"          # modelo, classes e gráficos
NOME_MODELO = "FaceCar_CNN_V3"
ARQUIVO_MODELO = "cnn_sonolencia"   # gera cnn_sonolencia.keras e cnn_sonolencia.classes.json

TAMANHO = 160                     # lado da imagem de entrada (o FaceCar_CNN_V2 também usa 160)
LOTE = 32                         # imagens por passo de treino
PROPORCOES = (0.70, 0.15, 0.15)   # treino, validação, teste
SEED = 42                         # mesma semente → mesma divisão e mesmo sorteio

EPOCAS = 30                  # máximo da fase 1; o EarlyStopping costuma parar antes
AJUSTE_FINO = True           # fase 2: libera as últimas camadas da MobileNetV2
EPOCAS_AJUSTE_FINO = 10
CAMADAS_LIBERADAS = 30       # quantas camadas do fim da MobileNetV2 voltam a aprender
PESOS_BASE = "imagenet"      # pesos pré-treinados da MobileNetV2

# Separação por pessoa (seção 4). Grupo 1 da regex = quem aparece na foto, tirado do NOME
# do arquivo. Ex.: r"^([a-z]+)_" faz "joao_0012.jpg" ser da pessoa "joao". None = por imagem.
REGEX_PESSOA = None

CACHE_NA_MEMORIA = True      # guarda as imagens já decodificadas na RAM (desligue se faltar memória)

## 2. Bibliotecas

Só o que é usado: **TensorFlow/Keras** (rede e leitura das imagens), **scikit-learn** (divisão dos dados e métricas), **NumPy** e **matplotlib**.

`keras.utils.set_random_seed` fixa de uma vez as sementes do Python, do NumPy e do TensorFlow. A GPU ainda pode dar diferenças minúsculas entre execuções, mas a divisão dos dados sai sempre igual.

In [ ]:
import importlib.util
import json
import re
import shutil
import time
from datetime import date
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(SEED)
EM_COLAB = importlib.util.find_spec("google.colab") is not None
NOMES = list(CLASSES)   # ["alerta", "sonolento", "bocejo"]: índice = saída do modelo

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "nenhuma (o treino vai ser lento)")
print("Colab:", EM_COLAB)

## 3. Copiar o dataset do Drive para o Colab

Ler milhares de arquivos pequenos direto do Drive é muito lento, porque cada leitura passa pela rede. A cópia para o disco local leva alguns minutos uma vez só, e o treino fica muitas vezes mais rápido.

Fora do Colab, esta célula não faz nada e usa `PASTA_LOCAL` direto.

In [ ]:
if EM_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    inicio = time.time()
    shutil.copytree(PASTA_DRIVE, PASTA_LOCAL, dirs_exist_ok=True)
    print(f"Cópia concluída em {(time.time() - inicio) / 60:.1f} min")
else:
    print("Fora do Colab: usando o dataset em", PASTA_LOCAL)

Path(PASTA_SAIDA).mkdir(parents=True, exist_ok=True)

## 4. Conferir o dataset e dividir em treino, validação e teste

Primeiro a célula confere se as subpastas de `CLASSES` existem e quantas imagens cada uma tem. Se uma classe tiver muito mais fotos que outra, a rede tende a "chutar" a maior. Por isso, na seção 7, cada classe ganha um **peso** inverso ao seu tamanho.

Depois vem a divisão em três partes:

- **treino**: as imagens com que a rede aprende;
- **validação**: o EarlyStopping olha para ela a cada época para decidir quando parar. Como ela influencia o treino, a nota dela é otimista;
- **teste**: fica guardado e só é usado **uma vez**, no fim. É a nota honesta.

**Cuidado com o vazamento de dados.** Datasets de sonolência costumam ser quadros de vídeo de poucas pessoas. Se a divisão for por imagem, dois quadros quase iguais da mesma pessoa caem um no treino e outro no teste. A rede acerta "reconhecendo a pessoa", e a nota fica perto de 100% sem valer para gente nova. Com `REGEX_PESSOA`, a divisão é **por pessoa**: cada uma fica inteira em um só conjunto.

A divisão é **estratificada** (sem `REGEX_PESSOA`): cada conjunto mantém a mesma proporção de alerta, sonolento e bocejo.

In [ ]:
EXTENSOES = {".jpg", ".jpeg", ".png", ".bmp"}

raiz = Path(PASTA_LOCAL)
subpastas = sorted(p.name for p in raiz.iterdir() if p.is_dir())
faltando = [pasta for pasta in CLASSES.values() if pasta not in subpastas]
if faltando:
    raise FileNotFoundError(f"Subpastas {faltando} não existem em {raiz}. Existem: {subpastas}. "
                            "Ajuste CLASSES na configuração.")
ignoradas = sorted(set(subpastas) - set(CLASSES.values()))
if ignoradas:
    print("Subpastas ignoradas (não estão em CLASSES):", ignoradas)

caminhos, rotulos = [], []
for indice, pasta in enumerate(CLASSES.values()):
    arquivos = sorted(str(f) for f in (raiz / pasta).rglob("*") if f.suffix.lower() in EXTENSOES)
    caminhos += arquivos
    rotulos += [indice] * len(arquivos)
caminhos, rotulos = np.array(caminhos), np.array(rotulos)

contagem = np.bincount(rotulos, minlength=len(NOMES))
for indice, (nome, pasta) in enumerate(CLASSES.items()):
    print(f"saída {indice} = {nome:<10} pasta '{pasta}': {contagem[indice]} imagens")
if contagem.min() == 0:
    raise ValueError("Há classe sem nenhuma imagem.")
if contagem.max() / contagem.min() > 1.5:
    print(f"Atenção: classes desbalanceadas (a maior tem {contagem.max() / contagem.min():.1f}x "
          "a menor). Os pesos por classe da seção 7 compensam.")

In [ ]:
_, FRACAO_VAL, FRACAO_TESTE = PROPORCOES
indices = np.arange(len(caminhos))

if REGEX_PESSOA:
    def pessoa(caminho):
        achou = re.search(REGEX_PESSOA, Path(caminho).name)
        if not achou:
            raise ValueError(f"REGEX_PESSOA não achou a pessoa em '{Path(caminho).name}'")
        return achou.group(1)

    grupos = np.array([pessoa(c) for c in caminhos])
    # Aqui as frações contam pessoas, não imagens.
    corte = GroupShuffleSplit(n_splits=1, test_size=FRACAO_TESTE, random_state=SEED)
    resto, idx_teste = next(corte.split(indices, rotulos, grupos))
    corte = GroupShuffleSplit(n_splits=1, test_size=FRACAO_VAL / (1 - FRACAO_TESTE),
                              random_state=SEED)
    i_treino, i_val = next(corte.split(resto, rotulos[resto], grupos[resto]))
    idx_treino, idx_val = resto[i_treino], resto[i_val]
    for nome, idx in [("treino", idx_treino), ("validação", idx_val), ("teste", idx_teste)]:
        print(f"{nome:<10} pessoas: {sorted(map(str, set(grupos[idx])))}")
else:
    resto, idx_teste = train_test_split(indices, test_size=FRACAO_TESTE, stratify=rotulos,
                                        random_state=SEED)
    idx_treino, idx_val = train_test_split(resto, test_size=FRACAO_VAL / (1 - FRACAO_TESTE),
                                           stratify=rotulos[resto], random_state=SEED)

assert not set(idx_treino) & set(idx_val) and not set(idx_treino) & set(idx_teste) \
    and not set(idx_val) & set(idx_teste), "uma imagem caiu em dois conjuntos"

print(f"\n{'':<10}" + "".join(f"{n:>11}" for n in NOMES) + f"{'total':>9}")
for nome, idx in [("treino", idx_treino), ("validação", idx_val), ("teste", idx_teste)]:
    por_classe = np.bincount(rotulos[idx], minlength=len(NOMES))
    print(f"{nome:<10}" + "".join(f"{n:>11}" for n in por_classe) + f"{len(idx):>9}")
    if por_classe.min() == 0:
        print(f"  Atenção: '{nome}' ficou sem alguma classe; ajuste PROPORCOES ou REGEX_PESSOA.")

## 5. Leitura das imagens (`tf.data`)

Cada imagem passa por estes passos:

1. lê o arquivo e decodifica em **RGB** (3 canais, mesmo se a foto for cinza);
2. redimensiona para `TAMANHO × TAMANHO` **sem recortar**, como `preparar()` faz no projeto com `cv2.resize`. Treino e uso precisam preparar a imagem do mesmo jeito;
3. arredonda para inteiros de 0 a 255, como a câmera entrega. Assim o cache ocupa 4× menos memória;
4. `cache`: depois da 1ª época, as imagens já vêm prontas da memória;
5. `shuffle` (só no treino): embaralha a cada época;
6. `batch`: junta em lotes de `LOTE`;
7. `prefetch`: prepara o próximo lote enquanto a GPU treina o atual.

Arquivo corrompido não derruba o treino: é pulado com um aviso (`ignore_errors`).

In [ ]:
def carregar_imagem(caminho, rotulo):
    imagem = tf.io.decode_image(tf.io.read_file(caminho), channels=3, expand_animations=False)
    imagem = tf.image.resize(imagem, (TAMANHO, TAMANHO))   # bilinear, sem recortar
    imagem = tf.cast(tf.clip_by_value(tf.round(imagem), 0, 255), tf.uint8)
    return imagem, rotulo


def para_float(imagem, rotulo):
    return tf.cast(imagem, tf.float32), rotulo   # o modelo recebe float32 de 0 a 255


def montar(indices, embaralhar):
    ds = tf.data.Dataset.from_tensor_slices((caminhos[indices], rotulos[indices]))
    ds = ds.map(carregar_imagem, num_parallel_calls=tf.data.AUTOTUNE).ignore_errors(log_warning=True)
    if CACHE_NA_MEMORIA:
        ds = ds.cache()
    if embaralhar:
        ds = ds.shuffle(len(indices), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(para_float).batch(LOTE).prefetch(tf.data.AUTOTUNE)


treino = montar(idx_treino, embaralhar=True)
validacao = montar(idx_val, embaralhar=False)
teste = montar(idx_teste, embaralhar=False)

imagens, rots = next(iter(treino))
plt.figure(figsize=(9, 9))
for i in range(min(9, len(imagens))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(imagens[i].numpy().astype("uint8"))
    plt.title(NOMES[int(rots[i])])
    plt.axis("off")
plt.suptitle("Amostras do treino (confira se o rótulo bate com a foto)")
plt.show()

## 6. O modelo

Uma imagem atravessa estas camadas:

| camada | o que faz | forma |
|---|---|---|
| `imagem_motorista` | entrada RGB, pixels de 0 a 255 | 160×160×3 |
| `aumento_dados` | **só no treino**: espelha, gira até ±10,8°, dá zoom, muda brilho e contraste | 160×160×3 |
| `Rescaling` | 0..255 → **−1..1**, a escala que a MobileNetV2 espera | 160×160×3 |
| `MobileNetV2` | extrai características visuais (congelada) | 5×5×1280 |
| `GlobalAveragePooling2D` | média de cada um dos 1280 mapas: resume a imagem em um vetor | 1280 |
| `Dropout(0.3)` | **só no treino**: desliga 30% das entradas ao acaso, para não decorar | 1280 |
| `classificacao` | `Dense` + `softmax`: 3 probabilidades que somam 1 | 3 |

**Aumento de dados.** A cada época a rede vê versões levemente diferentes de cada foto. Ela aprende, por exemplo, que olho fechado com a sala mais escura continua sendo olho fechado. Brilho e contraste imitam câmeras e iluminações diferentes. No uso real (`predict`), essas camadas ficam desligadas.

**Por que `base(x, training=False)`?** A MobileNetV2 tem camadas de *BatchNormalization*, que guardam médias aprendidas no ImageNet. Com `training=False`, elas continuam usando essas médias mesmo no ajuste fino. Sem isso, um lote pequeno bagunçaria as médias e estragaria a base.

In [ ]:
aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03, fill_mode="reflect"),        # 0,03 × 360° = ±10,8°
    layers.RandomZoom(0.05, fill_mode="reflect"),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
    layers.RandomContrast(0.15),
], name="aumento_dados")

base = keras.applications.MobileNetV2(weights=PESOS_BASE, include_top=False,
                                      input_shape=(TAMANHO, TAMANHO, 3))
base.trainable = False

entrada = keras.Input((TAMANHO, TAMANHO, 3), name="imagem_motorista")
x = aumento(entrada)
x = layers.Rescaling(1 / 127.5, offset=-1)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
saida = layers.Dense(len(NOMES), activation="softmax", name="classificacao")(x)
modelo = keras.Model(entrada, saida, name=NOME_MODELO)

modelo.summary()

## 7. Treino, fase 1: só a camada nova aprende

- **Adam** com taxa de aprendizado 0,001: o algoritmo que ajusta os pesos.
- **`sparse_categorical_crossentropy`**: a "nota de erro". Pune a rede quando ela dá probabilidade baixa à classe certa. "Sparse" porque os rótulos são inteiros (0, 1, 2).
- **Pesos por classe** (`class_weight`): errar uma classe com poucas fotos custa mais. Assim, a rede não ganha nota "chutando" a classe maior.
- **EarlyStopping**: se o erro na validação não melhorar em 4 épocas, para e volta aos pesos da melhor época.

In [ ]:
pesos = compute_class_weight("balanced", classes=np.arange(len(NOMES)), y=rotulos[idx_treino])
PESOS_CLASSE = dict(enumerate(pesos))
print("Pesos por classe:", {NOMES[i]: round(float(p), 2) for i, p in PESOS_CLASSE.items()})

modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

inicio = time.time()
hist_fase1 = modelo.fit(
    treino, validation_data=validacao, epochs=EPOCAS, class_weight=PESOS_CLASSE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                             restore_best_weights=True)])
print(f"Fase 1: {(time.time() - inicio) / 60:.1f} min")
historicos = [hist_fase1]

### Curvas de aprendizado

Como ler:

- **as duas curvas descendo juntas**: está aprendendo;
- **treino continua melhorando, validação piora**: *overfitting* (decoreba). O EarlyStopping corta aqui;
- **validação muito melhor que treino**: normal com Dropout e aumento de dados, que só agem no treino. Se for exagerado, desconfie de vazamento (seção 4).

In [ ]:
def mostrar_curvas(historicos, arquivo=None):
    juntar = lambda chave: sum((h.history[chave] for h in historicos), [])
    epocas = np.arange(1, len(juntar("loss")) + 1)
    fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
    for eixo, chave, titulo in [(eixos[0], "loss", "Erro (loss)"),
                                (eixos[1], "accuracy", "Acurácia")]:
        eixo.plot(epocas, juntar(chave), label="treino")
        eixo.plot(epocas, juntar("val_" + chave), label="validação")
        if len(historicos) > 1:
            eixo.axvline(len(historicos[0].history["loss"]) + 0.5, color="gray", linestyle="--",
                         label="início do ajuste fino")
        eixo.set_title(titulo)
        eixo.set_xlabel("época")
        eixo.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        eixo.legend()
    if arquivo:
        fig.savefig(arquivo, dpi=150, bbox_inches="tight")
    plt.show()


mostrar_curvas(historicos)

## 8. Treino, fase 2: ajuste fino (opcional)

Na fase 1 a MobileNetV2 ficou congelada. Agora as últimas `CAMADAS_LIBERADAS` camadas dela voltam a aprender. São as camadas que reconhecem padrões mais específicos, como "olho meio fechado", enquanto as primeiras (bordas, texturas) servem para qualquer foto.

A taxa de aprendizado cai 100 vezes (1e-5). Se fosse alta, a rede apagaria em poucos passos o que aprendeu no ImageNet. O `compile` é obrigatório depois de mudar o que é treinável.

Só vale a pena **depois** da fase 1. Se a camada nova ainda estiver aleatória, os erros grandes dela estragam a base.

In [ ]:
if AJUSTE_FINO:
    base.trainable = True
    for camada in base.layers[:-CAMADAS_LIBERADAS]:
        camada.trainable = False
    for camada in base.layers:
        if isinstance(camada, layers.BatchNormalization):
            camada.trainable = False   # as médias do ImageNet ficam como estão
    print("Camadas treináveis na base:", sum(c.trainable for c in base.layers))

    modelo.compile(optimizer=keras.optimizers.Adam(1e-5),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    inicio = time.time()
    hist_fase2 = modelo.fit(
        treino, validation_data=validacao, epochs=EPOCAS_AJUSTE_FINO, class_weight=PESOS_CLASSE,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                 restore_best_weights=True)])
    print(f"Fase 2: {(time.time() - inicio) / 60:.1f} min")
    historicos.append(hist_fase2)

mostrar_curvas(historicos, Path(PASTA_SAIDA) / "curvas_treino.png")

## 9. Avaliação no teste (a nota honesta)

Agora sim o teste é usado, e só esta vez. Para cada imagem, a classe prevista é a de maior probabilidade (`argmax`).

- **precision** de sonolento: das vezes que a rede disse "sonolento", quantas estavam certas (muitos alarmes falsos = precision baixa);
- **recall** de sonolento: dos motoristas realmente sonolentos, quantos ela pegou. **É a métrica mais importante aqui**, porque deixar passar alguém dormindo é muito pior que um alarme falso;
- **f1**: equilíbrio entre as duas;
- **matriz de confusão**: linha = classe real, coluna = classe prevista. A diagonal são os acertos, e o resto mostra *qual* classe é confundida com *qual*.

Imagens e rótulos são lidos **no mesmo laço**, para a previsão de cada foto ficar junto do rótulo dela.

In [ ]:
y_real, y_prob = [], []
for imagens, rots in teste:
    y_prob.append(modelo.predict_on_batch(imagens))
    y_real.append(rots.numpy())
y_real, y_prob = np.concatenate(y_real), np.concatenate(y_prob)
y_prev = y_prob.argmax(axis=1)

relatorio = classification_report(y_real, y_prev, labels=range(len(NOMES)), target_names=NOMES,
                                  digits=3, zero_division=0, output_dict=True)
print(classification_report(y_real, y_prev, labels=range(len(NOMES)), target_names=NOMES,
                            digits=3, zero_division=0))

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
matriz = confusion_matrix(y_real, y_prev, labels=range(len(NOMES)))
ConfusionMatrixDisplay(matriz, display_labels=NOMES).plot(ax=eixos[0], cmap="Blues",
                                                          values_format="d", colorbar=False)
eixos[0].set_title("Quantidade")
ConfusionMatrixDisplay(confusion_matrix(y_real, y_prev, labels=range(len(NOMES)), normalize="true"),
                       display_labels=NOMES).plot(ax=eixos[1], cmap="Blues", values_format=".0%",
                                                  colorbar=False)
eixos[1].set_title("Por linha (recall de cada classe)")
for eixo in eixos:
    eixo.set_xlabel("previsto")
    eixo.set_ylabel("real")
fig.suptitle(f"Matriz de confusão no teste - {NOME_MODELO}")
fig.savefig(Path(PASTA_SAIDA) / "matriz_confusao.png", dpi=150, bbox_inches="tight")
plt.show()

### E se o alarme disparar antes do 50%?

Com o `argmax`, a rede só diz "sonolento" quando essa é a maior probabilidade. Num sistema de segurança, pode valer a pena alertar mais cedo, por exemplo com P(sonolento) ≥ 30%: pega mais motoristas sonolentos (recall ↑) ao custo de mais alarmes falsos (precision ↓). A tabela mostra essa troca no teste e ajuda a escolher o limiar do projeto.

In [ ]:
SONOLENTO = NOMES.index("sonolento")
real = y_real == SONOLENTO
print(f"{'limiar':>7} {'recall':>8} {'precision':>10} {'alarmes':>8}")
for limiar in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    alarme = y_prob[:, SONOLENTO] >= limiar
    recall = (alarme & real).sum() / max(real.sum(), 1)
    precisao = (alarme & real).sum() / max(alarme.sum(), 1)
    print(f"{limiar:>7.0%} {recall:>8.1%} {precisao:>10.1%} {alarme.sum():>8}")

## 10. Salvar o modelo e a ordem das classes

São salvos dois arquivos:

- `cnn_sonolencia.keras`: arquitetura, pesos e as camadas de pré-processamento, tudo junto;
- `cnn_sonolencia.classes.json`: a ordem das saídas, o tamanho de entrada e as notas no teste. O projeto lê este arquivo e confere com `classificador.classes`.

Por fim, o modelo salvo é **recarregado** e a célula confere se ele prevê exatamente o mesmo que o da memória.

In [ ]:
saida = Path(PASTA_SAIDA)
caminho_modelo = saida / f"{ARQUIVO_MODELO}.keras"
caminho_classes = saida / f"{ARQUIVO_MODELO}.classes.json"

modelo.save(caminho_modelo)
caminho_classes.write_text(json.dumps({
    "modelo": NOME_MODELO,
    "classes": NOMES,
    "pastas": CLASSES,
    "entrada": [TAMANHO, TAMANHO, 3],
    "pixels": "RGB de 0 a 255, quadro inteiro redimensionado sem recorte",
    "divisao": "por pessoa" if REGEX_PESSOA else "por imagem",
    "teste": {
        "imagens": int(len(y_real)),
        "acuracia": round(relatorio["accuracy"], 4),
        "recall": {n: round(relatorio[n]["recall"], 4) for n in NOMES},
        "precision": {n: round(relatorio[n]["precision"], 4) for n in NOMES},
    },
    "treinado_em": date.today().isoformat(),
}, ensure_ascii=False, indent=2), encoding="utf-8")

recarregado = keras.models.load_model(caminho_modelo, compile=False)
lote = next(iter(teste))[0]
np.testing.assert_allclose(recarregado.predict_on_batch(lote), modelo.predict_on_batch(lote),
                           atol=1e-5)
print("Salvo e conferido:")
print(" ", caminho_modelo)
print(" ", caminho_classes)

if EM_COLAB:
    from google.colab import files

    files.download(str(caminho_modelo))
    files.download(str(caminho_classes))

## 11. Usar no projeto

1. Copie **os dois arquivos** (`cnn_sonolencia.keras` e `cnn_sonolencia.classes.json`) para a raiz do projeto.
2. Confira se `classificador.classes` em `config/analise.json` é `["alerta", "sonolento", "bocejo"]`. Se não for, o projeto avisa a diferença e não carrega o modelo.
3. Rode a análise ou o simulador normalmente: o tamanho de entrada é lido do próprio modelo.

**Limitações que continuam:**

- A rede recebe o **quadro inteiro** da câmera. Se as fotos do dataset forem rostos recortados, o que ela vê no uso é bem diferente do treino. O ideal é recortar o rosto nos dois lados.
- A câmera infravermelha entrega imagens em cinza, e o treino é com fotos coloridas.
- A nota do teste só vale para gente nova se a divisão for **por pessoa** (`REGEX_PESSOA`).